# Performance Tuning

This notebook covers performance optimization, benchmarking, and monitoring for the MCP Agent System.

## Key Metrics

- **Response time** - How fast tools execute
- **Throughput** - Concurrent task handling
- **Memory usage** - Resource consumption
- **Error rate** - Reliability metrics

In [ ]:
# Benchmarking utility tools
import asyncio
import time

from mcp_server.tools.utility_tools.calculator import CalculatorTool
from mcp_server.tools.utility_tools.text_processor import TextProcessorTool

async def benchmark():
    calc = CalculatorTool()
    txt = TextProcessorTool()
    
    # Calculator benchmark
    start = time.time()
    for i in range(100):
        await calc.execute(expression=f"{i} * 2")
    calc_time = time.time() - start
    print(f"Calculator: 100 calls in {calc_time:.3f}s ({100/calc_time:.0f} calls/s)")
    
    # Text processor benchmark
    start = time.time()
    for i in range(100):
        await txt.execute(text=f"test {i}", operation="uppercase")
    txt_time = time.time() - start
    print(f"TextProcessor: 100 calls in {txt_time:.3f}s ({100/txt_time:.0f} calls/s)")

asyncio.run(benchmark())

In [ ]:
# Concurrent execution test
import asyncio
import time

async def test_concurrency():
    tool = CalculatorTool()
    batch_sizes = [1, 5, 10, 20]
    
    for batch in batch_sizes:
        start = time.time()
        tasks = [tool.execute(expression=f"{i} + {i}") for i in range(batch)]
        results = await asyncio.gather(*tasks)
        elapsed = time.time() - start
        print(f"Batch size {batch}: {elapsed:.3f}s ({batch/elapsed:.0f} items/s)")
        assert len(results) == batch

asyncio.run(test_concurrency())

## Configuration Tuning

Key parameters to optimize:
- `max_steps`: Maximum reasoning steps per task
- `max_context_tokens`: Context window size
- `transport`: stdio vs HTTP (stdio has lower latency)
- Log level: INFO vs DEBUG (DEBUG adds overhead)

In [ ]:
# Configuration comparison
configs = [
    {"llm": "openai", "max_steps": 5, "max_context_tokens": 2000},
    {"llm": "openai", "max_steps": 10, "max_context_tokens": 4000},
    {"llm": "openai", "max_steps": 20, "max_context_tokens": 8000},
]
for i, cfg in enumerate(configs):
    print(f"Config {i+1}: max_steps={cfg['max_steps']}, context={cfg['max_context_tokens']}")

## Monitoring

Prometheus metrics are exposed at `/metrics` and Grafana dashboards provide visualization. Check the `monitoring/` directory for configuration.